In [22]:
#Imports and directory
import os
import pickle
import random
import math
import numpy as np
import pandas as pd
import matplotlib
import torch
import torch.nn as nn
from matplotlib import pyplot as plt
#os.chdir("/home/ec2-user/CS-230-Deep-Learning-Project")
os.chdir(r"C:\VScode\Projet Stanford CS230\CS-230-Deep-Learning-Project")
#os.chdir(r"C:\Users\gotta\OneDrive\Documents\Bureau\X\4A\US\Stanford\Classes\CS 230\Project\CS-230-Deep-Learning-Project")

In [23]:
# Répertoire
os.chdir(r"C:\VScode\Projet Stanford CS230\CS-230-Deep-Learning-Project")
data_dir = r"data\Datasets_2"

# Charger pkl
train_df = pd.read_pickle(os.path.join(data_dir, "train_set.pkl"))
dev_df   = pd.read_pickle(os.path.join(data_dir, "dev_set.pkl"))
test_df  = pd.read_pickle(os.path.join(data_dir, "test_set.pkl"))

In [27]:
import torch
import numpy as np

# Colonnes features et target
feature_cols = ['EUR/MWh','Load_DA','Wind_DA','Solar_DA','cos_hour','sin_hour', 'cos_day','sin_day', 'cos_month', 'sin_month']
target_col = 'pct'

def df_to_tensor(df, feature_cols):
    # Convert each column into a (N, 168) array, then stack along the last dimension
    arr = np.stack([np.vstack(df[col].values) for col in feature_cols], axis=2)
    return torch.tensor(arr, dtype=torch.float32)


X_train = df_to_tensor(train_df, feature_cols)
X_dev   = df_to_tensor(dev_df, feature_cols)
X_test  = df_to_tensor(test_df, feature_cols)

Y_train = torch.tensor(train_df[target_col].apply(lambda x: x[0]).values, dtype=torch.float32).unsqueeze(1)
Y_dev   = torch.tensor(dev_df[target_col].apply(lambda x: x[0]).values, dtype=torch.float32).unsqueeze(1)
Y_test  = torch.tensor(test_df[target_col].apply(lambda x: x[0]).values, dtype=torch.float32).unsqueeze(1)


# Vérification des shapes
print(X_train.shape, Y_train.shape)
print(X_dev.shape, Y_dev.shape)
print(X_test.shape, Y_test.shape)


torch.Size([63788, 168, 10]) torch.Size([63788, 1])
torch.Size([10264, 168, 10]) torch.Size([10264, 1])
torch.Size([27073, 168, 10]) torch.Size([27073, 1])


In [30]:
import torch
import torch.nn as nn

class ConvModel1(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv_layers = nn.Sequential(
            # ===== BLOCK 1 =====
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),   # 10 → 5
            nn.Dropout(0.1),

            # ===== BLOCK 2 =====
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),   # 5 → 2
            nn.Dropout(0.1),

            # ===== BLOCK 3 =====
            nn.Conv1d(128, 168, kernel_size=3, padding=1),
            nn.BatchNorm1d(168),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),   # 2 → 1
            nn.Dropout(0.1)
        )

        
        # Fully connected layers
        self.mlp = nn.Sequential(
            nn.Linear(168, 48),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(48, 24),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(24, 6),
            nn.ReLU(),

            nn.Linear(6, 3),
            nn.ReLU(),

            nn.Linear(3, 1)  # output
        )

    def forward(self, x):
        # x shape: (batch, 168, 10)
        x = x.permute(0, 2, 1)  # -> (batch, channels=10, length=168)

        x = self.conv_block(x)  # -> (batch, 168, 168)
        x = x.flatten(1)        # -> (batch, 168*168)
        x = self.fc(x)          # -> (batch, 1)

        return x


In [ ]:
import torch.optim as optim

# -------------------------
# Model
# -------------------------
model = ConvModel1()

# -------------------------
# Loss + Optimizer
# -------------------------
criterion = nn.MAELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# -------------------------
# Training parameters
# -------------------------
epochs = 200
batch_size = 256

# -------------------------
# Dataloaders
# -------------------------
train_dataset = torch.utils.data.TensorDataset(X_train, Y_train)
dev_dataset   = torch.utils.data.TensorDataset(X_dev, Y_dev)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dev_loader   = torch.utils.data.DataLoader(dev_dataset, batch_size=batch_size, shuffle=False)

# -------------------------
# Training loop
# -------------------------
train_losses = []
dev_losses = []

for epoch in range(epochs):

    # ===== TRAIN =====
    model.train()
    running_loss = 0.0
    for X, y in train_loader:
        optimizer.zero_grad()

        preds = model(X)
        loss = criterion(preds, y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # ===== VALIDATION =====
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X, y in dev_loader:
            preds = model(X)
            loss = criterion(preds, y)
            val_loss += loss.item()

    dev_loss = val_loss / len(dev_loader)
    dev_losses.append(dev_loss)

    print(f"Epoch {epoch+1}/{epochs} - Train: {train_loss:.5f} | Dev: {dev_loss:.5f}")


print("Training Finished.")

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_losses, label="Train Loss")
plt.plot(dev_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Curve")
plt.legend()
plt.grid(True)
plt.show()

In [31]:
def error_metric_torch(y_true, y_pred, shift=24*7, L1=True, Median=True):
    """
    Relative error vs a naive baseline in pure PyTorch.
    """

    y_true = y_true.flatten()
    y_pred = y_pred.flatten()

    # Align
    y_true_adj = y_true[shift:]
    y_pred_adj = y_pred[shift:]
    y_naive    = y_true[:-shift]

    # Baseline choice
    if Median:
        baseline = torch.median(y_true) if L1 else torch.mean(y_true)
        y_base = baseline.repeat(y_true_adj.shape[0])
    else:
        y_base = y_naive

    # Error type
    if L1:
        err_model = torch.mean(torch.abs(y_true_adj - y_pred_adj))
        err_base  = torch.mean(torch.abs(y_true_adj - y_base))
    else:
        err_model = torch.sqrt(torch.mean((y_true_adj - y_pred_adj)**2))
        err_base  = torch.sqrt(torch.mean((y_true_adj - y_base)**2))

    return (err_model / err_base).item()


In [32]:
# Define shifts
shifts = {
    "day-1": 24,        # j-1
    "week-1": 24*7      # week-1
}

# Metrics storage
metrics = {}

# List of datasets
datasets = {
    "train": (Y_train, model(X_train)),
    "dev":   (Y_dev, model(X_dev)),
    "test":  (Y_test, model(X_test))
}

# Compute metrics
for name, (y_true, y_pred) in datasets.items():
    metrics[name] = {}
    for shift_name, shift_val in shifts.items():
        # L1 vs naive
        rmae_naive = error_metric_torch(y_true, y_pred, shift=shift_val, L1=True, Median=False)
        # L1 vs median
        rmae_median = error_metric_torch(y_true, y_pred, shift=shift_val, L1=True, Median=True)
        metrics[name][shift_name] = {"naive": rmae_naive, "median": rmae_median}

# Display results
for dataset, vals in metrics.items():
    print(f"\n--- {dataset.upper()} ---")
    for shift_name, results in vals.items():
        print(f"{shift_name}: naive={results['naive']:.4f}, median={results['median']:.4f}")


NameError: name 'model' is not defined